# Stage 2 v2: 修复版 EKF Pruner

**第一版崩的根因**:
1. EKF没warm-up, 第一batch就剪
2. `tau0=0.3` 对初始 `theta=0` 太高, 所有gate都变0
3. `alpha=10` soft gate太sharp, 接近硬剪

**修复**:
1. Warm-up: 前N个batch只更新EKF, gate全开
2. 阈值相对化: `tau = base_ratio * max(theta)`, 按当前分布决定
3. `alpha=3.0` 让gate更柔
4. 初始`theta`不再全零, 用第一batch观测初始化

前提: 已挂载Drive, cwd = `/content/drive/MyDrive/ekf_adaptive_pruning`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ekf_adaptive_pruning')
print('[cwd]', os.getcwd())

## 覆盖写入 ekf_pruner.py (修复版)

In [ ]:
ekf_code = '''"""EKF-based channel-wise importance tracking and soft gating (v2)."""
import torch
import torch.nn as nn
import torch.nn.functional as F


def softmax_entropy(logits):
    p = F.softmax(logits, dim=-1)
    logp = F.log_softmax(logits, dim=-1)
    return -(p * logp).sum(dim=-1)


def max_softmax_prob(logits):
    p = F.softmax(logits, dim=-1)
    return 1.0 - p.max(dim=-1).values


class ChannelEKF:
    """Per-channel independent scalar EKF."""

    def __init__(self, num_channels, Q=1e-3, R=1e-2, device="cuda"):
        self.num_channels = num_channels
        self.Q = Q
        self.R = R
        self.device = device
        self.theta = None    # lazy init by first observation
        self.P = torch.full((num_channels,), 1.0, device=device)

    def update(self, obs):
        if self.theta is None:
            self.theta = obs.clone()     # init from first observation
            return
        P_pred = self.P + self.Q
        K = P_pred / (P_pred + self.R)
        self.theta = self.theta + K * (obs - self.theta)
        self.P = (1 - K) * P_pred


class EKFChannelGate(nn.Module):
    """Channel-wise soft gating with warm-up and relative threshold.

    Key design:
      - Warm-up: first `warmup_steps` batches only update EKF, gate=1
      - Relative threshold: tau = base_ratio * quantile(theta, q)
        where q scales with uncertainty (lower q = more aggressive prune)
      - Alpha moderate (3.0) for soft gating
    """

    def __init__(self, num_channels, Q=1e-3, R=1e-2, alpha=3.0,
                 prune_quantile_min=0.2, prune_quantile_max=0.5,
                 warmup_steps=5, device="cuda"):
        super().__init__()
        self.num_channels = num_channels
        self.alpha = alpha
        self.prune_quantile_min = prune_quantile_min   # aggressive (low uncertainty)
        self.prune_quantile_max = prune_quantile_max   # conservative (high uncertainty)
        self.warmup_steps = warmup_steps
        self.step = 0
        self.ekf = ChannelEKF(num_channels, Q=Q, R=R, device=device)
        self.current_uncertainty = 0.5
        self.last_gate = None
        self.last_keep_ratio = None

    def set_uncertainty(self, u):
        self.current_uncertainty = float(u)

    def forward(self, x):
        B, C, H, W = x.shape
        with torch.no_grad():
            obs = x.abs().mean(dim=[0, 2, 3])
        self.ekf.update(obs)
        self.step += 1

        # During warm-up, gate is fully open
        if self.step <= self.warmup_steps:
            self.last_gate = torch.ones(C, device=x.device)
            self.last_keep_ratio = 1.0
            return x

        # Relative threshold: prune bottom-q channels by importance
        # q interpolates based on uncertainty: low u -> aggressive (prune more)
        q = self.prune_quantile_min + (self.prune_quantile_max - self.prune_quantile_min) * (1 - self.current_uncertainty)
        q = max(0.0, min(0.9, q))
        tau = torch.quantile(self.ekf.theta, q).item()

        # Soft gate centered on tau
        gate = torch.sigmoid(self.alpha * (self.ekf.theta - tau) / (self.ekf.theta.std() + 1e-6))
        gate_bcast = gate.view(1, C, 1, 1)
        out = x * gate_bcast
        self.last_gate = gate.detach()
        self.last_keep_ratio = (gate > 0.5).float().mean().item()
        return out


class RandomChannelGate(nn.Module):
    def __init__(self, num_channels, keep_ratio=0.5, device="cuda"):
        super().__init__()
        self.num_channels = num_channels
        self.keep_ratio = keep_ratio
        self.last_keep_ratio = None

    def forward(self, x):
        B, C, H, W = x.shape
        n_keep = max(1, int(self.keep_ratio * C))
        perm = torch.randperm(C, device=x.device)
        mask = torch.zeros(C, device=x.device)
        mask[perm[:n_keep]] = 1.0
        self.last_keep_ratio = mask.mean().item()
        return x * mask.view(1, C, 1, 1)
'''

with open('models/ekf_pruner.py', 'w') as f:
    f.write(ekf_code)
print('ekf_pruner.py 覆盖完成:', os.path.getsize('models/ekf_pruner.py'), '字节')

## 覆盖 inference.py (修复: 兼容新API)

In [ ]:
inference_code = '''"""Inference with EKF-gated channel pruning (v2)."""
import os
import sys
import argparse

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

from models.resnet18_cifar import resnet18_cifar
from models.ekf_pruner import EKFChannelGate, RandomChannelGate, softmax_entropy


def get_test_loader(data_dir="./data", batch_size=128, num_workers=2):
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2470, 0.2435, 0.2616)
    test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    test_set = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=test_tf)
    return DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)


def wrap_layer_with_gate(layer_seq, gate_cls, gate_kwargs, device):
    new_modules = []
    gate_list = []
    for block in layer_seq:
        new_modules.append(block)
        out_ch = block.conv2.out_channels
        gate = gate_cls(num_channels=out_ch, device=device, **gate_kwargs).to(device)
        new_modules.append(gate)
        gate_list.append(gate)
    return nn.Sequential(*new_modules), gate_list


def install_gates(model, gate_cls, gate_kwargs, target_layers=("layer3", "layer4"), device="cuda"):
    all_gates = []
    for lname in target_layers:
        layer = getattr(model, lname)
        new_seq, gates = wrap_layer_with_gate(layer, gate_cls, gate_kwargs, device)
        setattr(model, lname, new_seq)
        all_gates.extend(gates)
    return all_gates


def evaluate_nogate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total


def evaluate_with_ekf_gates(model, gates, loader, device, entropy_norm=2.3):
    model.eval()
    correct, total = 0, 0
    keep_ratios = [[] for _ in gates]
    running_unc = 0.5
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            for g in gates:
                g.set_uncertainty(running_unc)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            ent = softmax_entropy(logits).mean().item()
            running_unc = min(1.0, ent / entropy_norm)
            for i, g in enumerate(gates):
                if g.last_keep_ratio is not None:
                    keep_ratios[i].append(g.last_keep_ratio)
    acc = 100.0 * correct / total
    avg_keep = [sum(r) / len(r) if r else 0.0 for r in keep_ratios]
    return acc, avg_keep


def evaluate_with_random_gates(model, gates, loader, device):
    model.eval()
    correct, total = 0, 0
    keep_ratios = [[] for _ in gates]
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            for i, g in enumerate(gates):
                if g.last_keep_ratio is not None:
                    keep_ratios[i].append(g.last_keep_ratio)
    acc = 100.0 * correct / total
    avg_keep = [sum(r) / len(r) if r else 0.0 for r in keep_ratios]
    return acc, avg_keep


def load_base_model(ckpt_path, device):
    model = resnet18_cifar(num_classes=10).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    return model


def run_comparison(ckpt_path, device="cuda"):
    loader = get_test_loader()

    print("\\n=== Baseline (no pruning) ===")
    m = load_base_model(ckpt_path, device)
    acc_base = evaluate_nogate(m, loader, device)
    print(f"acc = {acc_base:.2f}%")

    print("\\n=== EKF adaptive gating ===")
    m = load_base_model(ckpt_path, device)
    ekf_kwargs = dict(Q=1e-3, R=1e-2, alpha=3.0,
                      prune_quantile_min=0.2, prune_quantile_max=0.5,
                      warmup_steps=5)
    gates = install_gates(m, EKFChannelGate, ekf_kwargs,
                          target_layers=("layer3", "layer4"), device=device)
    acc_ekf, keep_ekf = evaluate_with_ekf_gates(m, gates, loader, device)
    avg_keep_ekf = sum(keep_ekf) / len(keep_ekf)
    print(f"acc = {acc_ekf:.2f}%,  avg_keep_ratio = {avg_keep_ekf:.3f}")
    print(f"per-gate keep: {[round(r, 3) for r in keep_ekf]}")

    # Cap keep ratio to avoid division issues
    match_keep = max(0.1, min(0.95, avg_keep_ekf))

    print("\\n=== Random gating (matched keep ratio) ===")
    m = load_base_model(ckpt_path, device)
    rand_kwargs = dict(keep_ratio=match_keep)
    gates = install_gates(m, RandomChannelGate, rand_kwargs,
                          target_layers=("layer3", "layer4"), device=device)
    acc_rand, keep_rand = evaluate_with_random_gates(m, gates, loader, device)
    avg_keep_rand = sum(keep_rand) / len(keep_rand)
    print(f"acc = {acc_rand:.2f}%,  avg_keep_ratio = {avg_keep_rand:.3f}")

    print("\\n=== Summary ===")
    print(f"  No pruning:      {acc_base:.2f}%")
    print(f"  EKF gating:      {acc_ekf:.2f}%  (keep {avg_keep_ekf:.2%})")
    print(f"  Random (match):  {acc_rand:.2f}%  (keep {avg_keep_rand:.2%})")
    print(f"  EKF advantage:   {acc_ekf - acc_rand:+.2f}% over random")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--ckpt", type=str, default="./checkpoints/resnet18_cifar_base.pth")
    args = parser.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[device] {device}")
    run_comparison(args.ckpt, device=device)
'''

with open('src/inference.py', 'w') as f:
    f.write(inference_code)
print('inference.py 覆盖完成:', os.path.getsize('src/inference.py'), '字节')

## 跑修复版对比

In [ ]:
!python src/inference.py

## 这次的预期

- **No pruning**: 94.97%
- **EKF gating**: 应该在 92-94% 之间 (keep 65-80%)
- **Random (match)**: 应该比 EKF 低 2-5%

关键看 `EKF advantage` 这一行, 只要是正的就说明机制work.

如果还有问题我们继续调超参.